In [0]:
%run ./00_config

In [0]:
#import libaries
from pyspark.sql import functions as F
import pandas as pd
import re
#functions to clean column name
def clean_col(c):
    c = c.strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c)
    return c.strip("_")
#function to clean area name
def clean_area(df):
    return (
        df.withColumn("area", F.lower(F.trim(F.col("area"))))
          .withColumn("area", F.regexp_replace("area", " council", ""))
          .withColumn("area", F.regexp_replace("area", "&", "and"))
          .withColumn("area", F.regexp_replace("area", r"\s+", " "))
    )

1. MSP Crime Data 


In [0]:
#reading data 
Mps_Raw = spark.read.option("header", True).option("inferSchema", True).csv(Path_for_MPS_Crime)
display(Mps_Raw.limit(8))

In [0]:
for old in Mps_Raw.columns:
    Mps_Raw= Mps_Raw.withColumnRenamed(old, clean_col(old))
print("Columns after cleaning",)
print(Mps_Raw.columns)


In [0]:
month_columns = [col for col in Mps_Raw.columns if re.match(r"^\d{6}$", col)]

print("Month columns found:", len(month_columns))
print(month_columns[:5])


In [0]:
#converting data types 
MPS_Long = Mps_Raw.select(
    F.col("boroughname").alias("area"),
    F.col("majortext").alias("major_category"),
    F.col("minortext").alias("minor_category"),
    F.explode(
        F.array(*[
            F.struct(
                F.lit(c).alias("period"),
                F.col(c).cast("double").alias("crime_count")
            )
            for c in month_columns
        ])
    ).alias("month_data")
)

MPS_Long = MPS_Long.select(
    "area",
    "major_category",
    "minor_category",
    F.col("month_data.period").alias("period"),
    F.col("month_data.crime_count").alias("crime_count")
)

display(MPS_Long.limit(9))

In [0]:
# Extracting months and years from period 
MPS_Long = MPS_Long.withColumn(
    "calendar_year",
    F.substring(F.col("period"), 1, 4).cast("int")
)

MPS_Long = MPS_Long.withColumn(
    "calendar_month",
    F.substring(F.col("period"), 5, 2).cast("int"))
display(MPS_Long.limit(5))

In [0]:
# 7. Converting month to financial year
MPS_Long = MPS_Long.withColumn(
    "fy_start",
    F.when(F.col("calendar_month") >= 4, F.col("calendar_year"))
     .otherwise(F.col("calendar_year") - 1)
)

MPS_Long = MPS_Long.withColumn(
    "financial_year",
    F.concat(
        F.col("fy_start").cast("string"),
        F.lit("/"),
        F.substring((F.col("fy_start") + 1).cast("string"), 3, 2)
    )
)

In [0]:
#filter out lower level crime 
crime_text = F.lower(F.concat_ws(" ", F.col("major_category"), F.col("minor_category")))

keywords = [
    "criminal damage", "arson", "theft", "shoplifting",
    "vehicle", "public order", "burglary", "robbery"
]

condition = None
for k in keywords:
    expr = crime_text.contains(k)
    condition = expr if condition is None else (condition | expr)

In [0]:
# 9. Aggregating  Borough and Financial year
MPS_silver = (
    MPS_Long
    .filter(condition)
    .filter(F.col("crime_count").isNotNull())
    .groupBy("area", "financial_year")
    .agg(F.sum("crime_count").alias("minor_crime_count"))
)

In [0]:
#Cleaning area names
MPS_silver = clean_area(MPS_silver)

# Saving table 
MPS_silver.write.mode("overwrite").format("delta").save(
    f"{Silver_Base}/Silver_MPS_minor_crime"
)

display(MPS_silver.orderBy("area", "financial_year").limit(20))

print("MPS Silver is saved successfully")

In [0]:
Raw_count = Mps_Raw.count()
Cln_count = MPS_silver.count()
print(f"MPS Raw rows: {Raw_count}")
print(f"MPS clean rows: {Cln_count}")
print(f"MPS rows dropped: {Raw_count - Cln_count}")

2. Fly-Tipping Data 


In [0]:
Fly_Raw = spark.read.option("header", True).option("inferSchema", True).csv(Path_for_fly_tipping)
display(Fly_Raw.limit(8))


In [0]:
for old in Fly_Raw.columns:
    Fly_Raw= Fly_Raw.withColumnRenamed(old, clean_col(old))
print("Columns after cleaning",)
print(Fly_Raw.columns)

In [0]:
Fly_Silver = (
    Fly_Raw.select(
        F.col("area").alias("area"),
        F.col("year").cast("string").alias("financial_year"),
        F.col("total_incidents").cast("string").alias("total_incidents_raw")
    )
    .withColumn("total_clean", F.regexp_replace("total_incidents_raw", ",", ""))
    .withColumn(
        "fly_tipping_incidents",
        F.when(F.col("total_clean") == ":", None)
         .otherwise(F.expr("try_cast(total_clean as double)"))
    )
    .filter(F.col("fly_tipping_incidents").isNotNull())
    .select("area", "financial_year", "fly_tipping_incidents")
)

Fly_Silver = clean_area(Fly_Silver)

Fly_Silver.write.mode("overwrite").format("delta").save(
    f"{Silver_Base}/silver_fly_tipping"
)

display(Fly_Silver.limit(10))

In [0]:
Raw_count = Fly_Raw.count()
Cln_count = Fly_Silver.count()
print(f"FLY Raw rows: {Raw_count}")
print(f"FLY clean rows: {Cln_count}")
print(f"FLY rows dropped: {Raw_count - Cln_count}")

3. CST Community Strength 

In [0]:
#reading data 
CST_Raw = spark.read.option("header", True).option("inferSchema", True).csv(Path_for_CST)
display(CST_Raw.limit(8))

In [0]:
#clean column name
for old in CST_Raw.columns:
    CST_Raw = CST_Raw.withColumnRenamed(old, clean_col(old))
print("Columns after cleaning",)
print(CST_Raw.columns)

In [0]:
# Rename columns
CST_silver = CST_Raw.select(
    F.col("la").alias("area"),
    F.col("percentage_of_adults_16_reporting_that_people_in_neighbourhood_pull_together_to_improve_the_neighbourhood")
        .alias("neighbour_all_together"),

    F.col("percentage_of_adults_16_reporting_that_if_they_needed_help_there_are_people_who_would_be_there_for_them")
        .alias("people_help_avail"),

    F.col("index_of_local_area_belonging")
        .alias("local_area_belong_index"),

    F.col("percentage_of_adults_16_reporting_taking_part_in_formal_volunteering_at_least_once_in_the_last_year")
        .alias("formal_volunt_percentage"),

    F.col("percentage_of_adults_16_reporting_that_they_felt_lonely_often_or_always")
        .alias("lonely_percentage")

)
display(CST_silver.limit(6))

In [0]:
CST_silver = clean_area(CST_silver)
CST_silver.write.mode("overwrite").format("delta").save(
    f"{Silver_Base}/silver_community_strength"
)

display(CST_silver.limit(10))

print("CST created successfully ")

In [0]:
Raw_count =CST_Raw.count()
Cln_count = CST_silver.count()
print(f"CST Raw rows: {Raw_count}")
print(f"CST clean rows: {Cln_count}")
print(f"CST rows dropped: {Raw_count - Cln_count}")